# PhytopathAI pipeline: full rebuild (one notebook)

**Before you start:** Runtime → Change runtime type → **T4 GPU**. Then Runtime → **Run all**.
It takes roughly 20–40 minutes. When it finishes, the last cell downloads `thesis_results.zip`. Send that file back.

Expects your BACH images in Google Drive at `MyDrive/BreastCancerProject/Train/{Normal,Benign,InSitu,Invasive}/*.tif` (the same place your original notebooks used). Everything this run writes goes to a new folder, `MyDrive/BreastCancerProject/rebuild_v3/`, so nothing from your earlier runs is overwritten.

### What changed from the original Phase1–Phase4 notebooks, and why
| Change | Why |
|---|---|
| VGG16 `preprocess_input` applied | The original fed raw 0–255 pixels to VGG16; its ImageNet weights expect mean-centred BGR input. |
| Separate validation set (15% of training images) for early stopping and checkpoint choice; test set used once at the end | The original chose the best epoch by **test** accuracy, which inflates the reported number. |
| Each training stage has its own early-stopping callback | The original reused one callback, so its state carried from stage 1 into stage 2. |
| 95% bootstrap confidence interval for test accuracy | 80 test images give a wide margin of error; the interval shows it. |
| Cluster file read from the folder it is written to | Script 03 looked in `Outputs/`; script 02 wrote to `models/`. It would have crashed. |
| Phytochemical table with checked PubMed links (embedded below) | The original PubMed IDs did not point to the claimed studies. |

Kept as in the original on purpose: 224×224 whole-image resize, batch size 16, last 4 VGG16 layers unfrozen, epochs 40 + 20, ResNet50 + StandardScaler + K-Means (k=2), the nuclear and glandular morphology functions.

In [ ]:
import os, json, shutil, random
os.environ["TF_DETERMINISTIC_OPS"] = "1"
from google.colab import drive
drive.mount("/content/drive")

import numpy as np, pandas as pd, cv2, matplotlib
import matplotlib.pyplot as plt
import tensorflow as tf
print("TensorFlow", tf.__version__, "| GPU:", tf.config.list_physical_devices("GPU") or "NONE - switch to a T4 GPU runtime")

PROJECT = "/content/drive/MyDrive/BreastCancerProject"
RAW_DIR = f"{PROJECT}/Train"
OUT = f"{PROJECT}/rebuild_v3"
DATA, MODELS, RESULTS = f"{OUT}/data", f"{OUT}/models", f"{OUT}/results"
for d in (DATA, MODELS, RESULTS): os.makedirs(d, exist_ok=True)

CLASS_MAP = {"Normal": 0, "Benign": 1, "InSitu": 2, "Invasive": 3}
CLASS_NAMES = list(CLASS_MAP)
IMG_SIZE = (224, 224)
SEED = 42
random.seed(SEED); np.random.seed(SEED); tf.random.set_seed(SEED)

counts = {c: len([f for f in os.listdir(f"{RAW_DIR}/{c}") if f.lower().endswith((".tif", ".tiff"))]) for c in CLASS_NAMES}
print("Images per class:", counts)
assert all(v > 0 for v in counts.values()), f"Missing images - check {RAW_DIR}"

## 1. Load images and split (train / validation / test)

In [ ]:
from sklearn.model_selection import train_test_split

images, labels, names = [], [], []
for cname, lab in CLASS_MAP.items():
    for fn in sorted(os.listdir(f"{RAW_DIR}/{cname}")):
        if fn.lower().endswith((".tif", ".tiff")):
            img = cv2.cvtColor(cv2.imread(f"{RAW_DIR}/{cname}/{fn}"), cv2.COLOR_BGR2RGB)
            images.append(cv2.resize(img, IMG_SIZE)); labels.append(lab); names.append(f"{cname}/{fn}")
images, labels, names = np.array(images), np.array(labels), np.array(names)

# Test set: 20%, same seed and stratification as the original, so it is the same 80 images.
idx = np.arange(len(labels))
trainval_idx, test_idx = train_test_split(idx, test_size=0.2, random_state=SEED, stratify=labels)
# New: validation set carved from the training images only.
train_idx, val_idx = train_test_split(trainval_idx, test_size=0.15, random_state=SEED, stratify=labels[trainval_idx])
print(f"train {len(train_idx)} | validation {len(val_idx)} | test {len(test_idx)}")
pd.DataFrame({"file": names[test_idx], "label": labels[test_idx]}).to_csv(f"{RESULTS}/test_set_files.csv", index=False)

# Lesion-only set for clustering (Benign, InSitu, Invasive), 70/30, as in the original.
lesion_idx = np.where(labels >= CLASS_MAP["Benign"])[0]
Xc_train, Xc_val, yc_train, yc_val = train_test_split(
    images[lesion_idx], labels[lesion_idx], test_size=0.3, random_state=SEED, stratify=labels[lesion_idx])
print(f"clustering: train {len(yc_train)} | held-out {len(yc_val)}")

## 2. Train the VGG16 classifier (two stages)

In [ ]:
from tensorflow.keras.applications import VGG16
from tensorflow.keras.applications.vgg16 import preprocess_input
from tensorflow.keras.layers import Dense, Flatten, Dropout, Input
from tensorflow.keras.models import Model
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras.callbacks import ModelCheckpoint, EarlyStopping
from tensorflow.keras.utils import to_categorical

prep = lambda a: preprocess_input(a.astype("float32").copy())
X_tr, X_va, X_te = prep(images[train_idx]), prep(images[val_idx]), prep(images[test_idx])
y_tr, y_va, y_te = (to_categorical(labels[i], 4) for i in (train_idx, val_idx, test_idx))

aug = ImageDataGenerator(rotation_range=10, width_shift_range=0.1, height_shift_range=0.1,
                         shear_range=0.1, zoom_range=0.1, horizontal_flip=True, fill_mode="nearest")
BATCH = 16

base = VGG16(weights="imagenet", include_top=False, input_tensor=Input(shape=(*IMG_SIZE, 3)))
base.trainable = False
x = Dropout(0.5)(Dense(256, activation="relu")(Flatten()(base.output)))
model = Model(base.input, Dense(4, activation="softmax")(x))

def run_stage(epochs, lr, ckpt):
    model.compile(optimizer=Adam(learning_rate=lr), loss="categorical_crossentropy", metrics=["accuracy"])
    return model.fit(
        aug.flow(X_tr, y_tr, batch_size=BATCH, seed=SEED),
        steps_per_epoch=len(X_tr) // BATCH, epochs=epochs,
        validation_data=(X_va, y_va),  # validation set, never the test set
        callbacks=[ModelCheckpoint(ckpt, monitor="val_loss", save_best_only=True, verbose=1),
                   EarlyStopping(monitor="val_loss", patience=10, restore_best_weights=True, verbose=1)])

print("=== Stage 1: train the new head (VGG16 frozen) ===")
h1 = run_stage(40, 1e-3, f"{MODELS}/classifier_stage1.keras")

print("=== Stage 2: fine-tune the last 4 VGG16 layers ===")
base.trainable = True
for layer in base.layers[:-4]: layer.trainable = False
h2 = run_stage(20, 1e-5, f"{MODELS}/classifier_final.keras")

for h, tag in ((h1, "stage1"), (h2, "stage2")):
    pd.DataFrame(h.history).to_csv(f"{RESULTS}/training_history_{tag}.csv", index_label="epoch")

## 3. Evaluate once on the 80 test images

In [ ]:
from sklearn.metrics import classification_report, confusion_matrix, ConfusionMatrixDisplay, matthews_corrcoef, cohen_kappa_score

final = tf.keras.models.load_model(f"{MODELS}/classifier_final.keras")
probs = final.predict(X_te, verbose=0)
y_pred, y_true = probs.argmax(1), y_te.argmax(1)
acc = float((y_pred == y_true).mean())

rng = np.random.default_rng(SEED)
boot = [(y_pred[s] == y_true[s]).mean() for s in (rng.integers(0, len(y_true), len(y_true)) for _ in range(2000))]
lo, hi = np.percentile(boot, [2.5, 97.5])

ConfusionMatrixDisplay(confusion_matrix(y_true, y_pred), display_labels=CLASS_NAMES).plot(cmap="Blues", values_format="d")
plt.title("Test set confusion matrix"); plt.savefig(f"{RESULTS}/confusion_matrix.png", dpi=200, bbox_inches="tight"); plt.show()

pd.DataFrame(classification_report(y_true, y_pred, target_names=CLASS_NAMES, output_dict=True)).T.round(4) \
  .to_csv(f"{RESULTS}/classification_report.csv")
pd.DataFrame(probs, columns=CLASS_NAMES).assign(true=y_true, pred=y_pred, file=names[test_idx]) \
  .to_csv(f"{RESULTS}/test_predictions.csv", index=False)

summary = {
    "test_images": int(len(y_true)), "test_accuracy": acc, "accuracy_95ci": [float(lo), float(hi)],
    "macro_f1": float(classification_report(y_true, y_pred, output_dict=True)["macro avg"]["f1-score"]),
    "matthews_corrcoef": float(matthews_corrcoef(y_true, y_pred)),
    "cohen_kappa": float(cohen_kappa_score(y_true, y_pred)),
    "stage1_epochs": len(h1.history["loss"]), "stage2_epochs": len(h2.history["loss"]),
    "train_images": int(len(train_idx)), "validation_images": int(len(val_idx)),
    "original_reported_accuracy": 0.7875,
    "note": "Original 78.75% used no VGG16 preprocessing and picked the best epoch on the test set.",
}
json.dump(summary, open(f"{RESULTS}/training_summary.json", "w"), indent=2)
print(json.dumps(summary, indent=2))
print(classification_report(y_true, y_pred, target_names=CLASS_NAMES, digits=4))

## 4. ResNet50 features and K-Means clustering of lesion images

In [ ]:
import joblib
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score, calinski_harabasz_score, davies_bouldin_score, adjusted_rand_score
from sklearn.manifold import TSNE
from scipy.stats import chi2_contingency
from tensorflow.keras.applications import ResNet50
from tensorflow.keras.applications.resnet50 import preprocess_input as resnet_prep

extractor = ResNet50(weights="imagenet", include_top=False, pooling="avg")
feats = lambda a: extractor.predict(resnet_prep(a.astype("float32").copy()), batch_size=16, verbose=0)
F_train, F_val = feats(Xc_train), feats(Xc_val)

scaler = StandardScaler().fit(F_train)
S_train, S_val = scaler.transform(F_train), scaler.transform(F_val)
joblib.dump(scaler, f"{MODELS}/feature_scaler.pkl")
np.save(f"{MODELS}/train_features_scaled.npy", S_train)

rows = []
for k in range(2, 6):
    km = KMeans(n_clusters=k, random_state=SEED, n_init=10).fit(S_train)
    rows.append({"k": k, "silhouette": silhouette_score(S_train, km.labels_),
                 "calinski_harabasz": calinski_harabasz_score(S_train, km.labels_),
                 "davies_bouldin": davies_bouldin_score(S_train, km.labels_)})
kdf = pd.DataFrame(rows).round(4); kdf.to_csv(f"{RESULTS}/cluster_k_selection.csv", index=False); print(kdf)

K = 2  # as in the original
kmeans = KMeans(n_clusters=K, random_state=SEED, n_init=10).fit(S_train)
joblib.dump(kmeans, f"{MODELS}/kmeans_model.pkl")
c_train, c_val = kmeans.labels_, kmeans.predict(S_val)

DIAG = {1: "Benign", 2: "InSitu", 3: "Invasive"}
ct = pd.crosstab(pd.Series(c_train, name="cluster"), pd.Series(yc_train, name="diagnosis").map(DIAG))
ct_pct = (ct.div(ct.sum(1), axis=0) * 100).round(2)
ct.to_csv(f"{RESULTS}/cluster_diagnostic_crosstab.csv"); ct_pct.to_csv(f"{RESULTS}/cluster_diagnostic_crosstab_pct.csv")
ct_val = pd.crosstab(pd.Series(c_val, name="cluster"), pd.Series(yc_val, name="diagnosis").map(DIAG))
ct_val.to_csv(f"{RESULTS}/cluster_diagnostic_crosstab_heldout.csv")

validation = {
    "k": K,
    "train_silhouette": float(silhouette_score(S_train, c_train)),
    "heldout_silhouette": float(silhouette_score(S_val, c_val)),
    "train_chi2_p": float(chi2_contingency(ct)[1]),
    "heldout_chi2_p": float(chi2_contingency(ct_val)[1]),
    "train_adjusted_rand_vs_diagnosis": float(adjusted_rand_score(yc_train, c_train)),
    "note": "Silhouette near 0 means clusters overlap heavily. A low chi-square p only means cluster membership "
            "is associated with diagnosis.",
}
json.dump(validation, open(f"{RESULTS}/validation_summary.json", "w"), indent=2)
print(ct); print(json.dumps(validation, indent=2))

emb = TSNE(n_components=2, random_state=SEED, perplexity=30).fit_transform(S_train)
fig, ax = plt.subplots(1, 2, figsize=(13, 5.5))
for a, col, title in ((ax[0], c_train, "K-Means cluster"), (ax[1], yc_train, "True diagnosis (1 Benign, 2 InSitu, 3 Invasive)")):
    sc = a.scatter(emb[:, 0], emb[:, 1], c=col, cmap="viridis", s=18, alpha=.8); a.set_title(title); plt.colorbar(sc, ax=a)
plt.tight_layout(); plt.savefig(f"{RESULTS}/tsne_clusters_vs_diagnosis.png", dpi=200); plt.show()

## 5. Nuclear and glandular morphology per cluster (functions from Phase4)

In [ ]:
from scipy import stats
from skimage import color, exposure, feature, measure, morphology

def nuclear_morphology(rgb):
    g = exposure.equalize_adapthist(color.rgb2gray(rgb))
    t = morphology.binary_opening(morphology.binary_closing(g < 0.65, morphology.disk(1)), morphology.disk(1))
    areas = [r.area for r in measure.regionprops(measure.label(t)) if 10 < r.area < 200]
    return (np.median(areas), np.std(areas)) if areas else (np.nan, np.nan)

def glandular_score(rgb):
    return np.mean(morphology.binary_closing(feature.canny(color.rgb2gray(rgb), sigma=1.5), morphology.disk(3)))

rng = np.random.default_rng(SEED); mrows = []
for cid in range(K):
    members = np.where(c_train == cid)[0]
    for i in rng.choice(members, min(100, len(members)), replace=False):
        size, pleo = nuclear_morphology(Xc_train[i])
        mrows.append({"cluster": cid, "diagnosis": DIAG[int(yc_train[i])], "nuclear_size": size,
                      "nuclear_pleomorphism": pleo, "glandular_score": glandular_score(Xc_train[i])})
morph = pd.DataFrame(mrows).dropna(); morph.to_csv(f"{RESULTS}/morphology_per_image.csv", index=False)

pvals = {f: float(stats.mannwhitneyu(morph[morph.cluster == 0][f], morph[morph.cluster == 1][f])[1])
         for f in ("nuclear_size", "nuclear_pleomorphism", "glandular_score")}
msum = morph.groupby("cluster")[["nuclear_size", "nuclear_pleomorphism", "glandular_score"]].agg(["median", "std"]).round(3)
msum = msum.reindex(range(K))  # keeps a row (NaN) for a cluster where no nuclei were found, instead of crashing
print(f"Images with measurable nuclei: {len(morph)} of {len(mrows)}")
msum.to_csv(f"{RESULTS}/cluster_morphology_summary.csv")
print(msum); print("Mann-Whitney p-values:", pvals)
print("Note: these are measured on 224x224 downsized images, so 'nuclei' are only a few pixels across.")

defs = {}
for cid in range(K):
    comp = ct_pct.loc[cid].sort_values(ascending=False)
    defs[str(cid)] = {"diagnostic_composition_pct": comp.to_dict(), "dominant_diagnosis": comp.index[0],
                      "n_images": int((c_train == cid).sum()),
                      **{f"median_{f}": float(msum.loc[cid, (f, "median")]) for f in pvals},
                      "morphology_p_values": pvals, **validation}
json.dump(defs, open(f"{MODELS}/cluster_definitions.json", "w"), indent=2)
print(json.dumps(defs, indent=2))

## 6. Map clusters to phytochemicals (checked literature table)

In [ ]:
import io
PHYTO_DB_CSV = """compound,source_plant,target_cluster,cluster_anchor_rationale,mechanism,evidence_tier,study_design,citation_title,pubmed_url
Genistein,Soy (Glycine max),Early-lesion-enriched,"Cluster 0 diagnostic composition is Benign+In Situ (early/pre-invasive lesions, p<0.001 vs random); genistein's ER-beta-mediated action targets hormone-driven proliferative signaling relevant to this stage.","ER-beta-mediated modulation of estrogen signaling; inhibits proliferation, invasion and migration; induces G2/M arrest and apoptosis. Effects reported as dose- and menopausal-status-dependent.",B,Review synthesizing multiple in vitro breast cancer cell line studies,Genistein: A Potent Anti-Breast Cancer Agent,https://pubmed.ncbi.nlm.nih.gov/34698063/
Indole-3-Carbinol,"Cruciferous vegetables (broccoli, cabbage)",Early-lesion-enriched,"Cluster 0 is enriched for Benign/In Situ lesions; I3C's mechanism (shifting estrogen metabolism away from proliferative signaling) is specifically relevant to early, hormone-responsive disease stages rather than established invasive disease.",Shifts estradiol metabolism toward non-estrogenic 2-hydroxylated metabolites; abrogates estrogen-receptor-mediated proliferative signaling in estradiol-responsive breast cancer cell lines.,B,In vitro mechanistic studies; human urinary estrogen-metabolite data available (separate PMID 1656396) but not cancer-outcome data,Inhibition of proliferation and modulation of estradiol metabolism: novel mechanisms for breast cancer prevention by the phytochemical indole-3-carbinol,https://pubmed.ncbi.nlm.nih.gov/9349693/
Apigenin,"Parsley, chamomile",Early-lesion-enriched,"Cluster 0's early/pre-invasive composition matches apigenin's documented activity in ER-positive, less-aggressive breast cancer contexts (proliferation inhibition rather than anti-metastatic action).",Partial estrogen-receptor antagonist; inhibits ER-positive proliferation via Akt/FOXM1 signaling; in HER2-overexpressing lines induces p53-dependent apoptosis via the extrinsic pathway and inhibits STAT3/NF-kB signaling.,B,Multiple in vitro breast cancer cell line studies,Apigenin induces apoptosis via extrinsic pathway inducing p53 and inhibiting STAT3 and NFkB signaling in HER2-overexpressing breast cancer cells,https://pubmed.ncbi.nlm.nih.gov/22527937/
Curcumin,Turmeric (Curcuma longa),Invasive-enriched,"Cluster 1 diagnostic composition is 67.4% Invasive Carcinoma (p<0.001 vs random); curcumin's documented anti-metastatic, NF-kB-suppressing mechanism directly targets the invasion/metastasis biology this cluster is enriched for.","Suppresses NF-kB-mediated urokinase-type plasminogen activator (uPA) expression, inhibiting adhesion, invasion and metastatic progression; downregulates anti-apoptotic (Bcl-2, Bcl-xL, XIAP) and proliferative (COX-2, c-Myc, cyclin D1) protein expression.",A,"In vitro + in vivo (nude mouse lung metastasis model); a registered Phase II RCT (NCT01740323) exists for curcumin in chemotherapy-treated breast cancer patients, though that trial evaluates radiotherapy-related outcomes, not this specific mechanism",Curcumin inhibits metastatic progression of breast cancer cell through suppression of urokinase-type plasminogen activator by NF-kappa B signaling pathways,https://pubmed.ncbi.nlm.nih.gov/21947854/
Sulforaphane,Broccoli sprouts (Brassica oleracea),Invasive-enriched,Cluster 1's invasive-enriched composition matches sulforaphane's documented activity against aggressive/stem-like breast cancer phenotypes and metastatic spread.,"Induces cell-type-specific apoptosis (G2/M block, cyclin B1 upregulation, DNA fragmentation); depletes ALDH-positive breast cancer stem cell population; anti-metastatic activity reported in triple-negative breast cancer models.",A,In vitro (multiple cell lines) + in vivo xenograft ALDH reduction,Sulforaphane induces cell type-specific apoptosis in human breast cancer cell lines,https://pubmed.ncbi.nlm.nih.gov/17339367/
Withaferin A,Ashwagandha (Withania somnifera),Invasive-enriched,"Cluster 1 is dominated by invasive carcinoma; Withaferin A's mechanism (direct disruption of the vimentin cytoskeleton, the structural driver of EMT/metastatic spread) is specifically anti-invasive/anti-metastatic rather than broadly cytotoxic.","Induces vimentin cytoskeleton disassembly (perinuclear accumulation + serine-56 phosphorylation) at sub-cytotoxic doses, directly inhibiting invasion; suppresses NF-kB/STAT3 signaling and breast cancer stem cell self-renewal; reduces metastatic lung nodules in vivo with minimal toxicity.",A,In vitro + in vivo mouse metastasis model,Withaferin A inhibits breast cancer invasion and metastasis at sub-cytotoxic doses by inducing vimentin disassembly and serine 56 phosphorylation,https://pubmed.ncbi.nlm.nih.gov/21538350/
Resveratrol,"Grapes, berries",Invasive-enriched,"Cluster 1's invasive/aggressive composition aligns with resveratrol's documented pro-apoptotic action in breast cancer cells, though its effect is p53-status-dependent — a caveat this project cannot resolve without molecular subtyping.","Induces p53-dependent apoptosis via ERK/p38-mediated phosphorylation of p53; effect demonstrated specifically in wild-type p53-expressing breast cancer cells, not in p53-mutant lines.",B,"In vitro; mechanism is conditional on p53 mutation status, which this project's histology-only pipeline cannot determine",Identification of a p53-dependent pathway in the induction of apoptosis of human breast cancer cells by the natural product resveratrol,https://pubmed.ncbi.nlm.nih.gov/15165403/
Epigallocatechin gallate (EGCG),Green tea (Camellia sinensis),Both,"Relevant to both clusters: EGCG's multi-pathway action (metabolic suppression, anti-angiogenic, anti-migratory) is documented across both early proliferative and invasive/triple-negative breast cancer contexts, unlike the other compounds which are stage-specific.","Suppresses glycolytic metabolism (inhibits hexokinase, phosphofructokinase, lactate dehydrogenase; downregulates HIF1-alpha/GLUT1); inhibits VEGF-driven angiogenesis and migration; induces caspase-dependent apoptosis; modulates Nrf2/NF-kB signaling.",A,In vitro (including triple-negative lines) + preclinical glucose-metabolism models,Epigallocatechin-3-Gallate (EGCG) inhibits cell proliferation and migratory behaviour of triple negative breast cancer cells,https://pubmed.ncbi.nlm.nih.gov/23646788/
"""
phyto = pd.read_csv(io.StringIO(PHYTO_DB_CSV))
phyto.to_csv(f"{RESULTS}/phytochemical_db_v2.csv", index=False)

TO_CATEGORY = {"Invasive": "Invasive-enriched", "Benign": "Early-lesion-enriched", "InSitu": "Early-lesion-enriched"}
recs = []
for cid, info in defs.items():
    cat = TO_CATEGORY[info["dominant_diagnosis"]]
    m = phyto[(phyto.target_cluster == cat) | (phyto.target_cluster == "Both")].sort_values("evidence_tier")
    recs += [{"cluster": cid, "cluster_category": cat, **r} for r in m.to_dict("records")]
if len({TO_CATEGORY[i["dominant_diagnosis"]] for i in defs.values()}) < K:
    print("WARNING: both clusters have the same dominant diagnosis, so they get the same compounds. Report this.")
recs = pd.DataFrame(recs); recs.to_csv(f"{RESULTS}/phytochemical_recommendations_final.csv", index=False)
print(recs[["cluster", "cluster_category", "compound", "source_plant", "evidence_tier"]].to_string(index=False))

## 7. Download the results

In [ ]:
shutil.copy(f"{MODELS}/cluster_definitions.json", RESULTS)
zip_path = shutil.make_archive("/content/thesis_results", "zip", RESULTS)
print("Results folder also saved in Drive:", RESULTS)
print("Trained models (not in the zip, too large) are in Drive:", MODELS)
from google.colab import files
files.download(zip_path)

## 8. Export the small files the PhytopathAI app uses
Half-precision weights (about 39 MB; on the 80 test images they give the same predictions as the full model) and the clustering parameters as plain arrays, so the app needs no pickles.

In [ ]:
APP = f"{OUT}/app"; os.makedirs(APP, exist_ok=True)
ws = final.get_weights()
np.savez_compressed(f"{APP}/phytopathai_vgg16_fp16.npz", *[w.astype(np.float16) for w in ws])
check = tf.keras.models.clone_model(final)
check.set_weights([w.astype(np.float16).astype(np.float32) for w in ws])
same = int((check.predict(X_te, verbose=0).argmax(1) == y_pred).sum())
print(f"fp16 weights give the same test prediction for {same} of {len(y_pred)} images")

np.savez(f"{APP}/cluster_params.npz", scaler_mean=scaler.mean_, scaler_scale=scaler.scale_,
         centers=kmeans.cluster_centers_,
         ood_threshold_p95=np.percentile(kmeans.transform(S_train).min(axis=1), 95))
files.download(f"{APP}/cluster_params.npz")